# Transportation Telemetry Predictive Timing Demo

## Project Question

**Can a real-time transportation telemetry platform predict train delay timing and support predictive maintenance decision-making using sensor, location, and operational event data?**

This notebook demonstrates:
1. Synthetic train telemetry generation
2. Bronze / Silver / Gold data engineering pipeline
3. Neural network predictive timing model
4. Delay risk classification and delay-minute regression
5. Evaluation metrics and visual outputs
6. Architecture interpretation for a Sr/Staff Data Engineer portfolio


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

print("Project root:", ROOT)

Project root: /Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing


## 1. Generate synthetic telemetry data

In [2]:
from src.data_generator import generate_synthetic_telemetry

df = generate_synthetic_telemetry(
    n_events=5000,
    output_path=ROOT / "data/raw/train_telemetry_events.csv"
)

display(df.head())
print(df.shape)

,event_id,event_time,train_id,route_id,latitude,longitude,scheduled_minutes,distance_miles,avg_speed_mph,brake_pressure,engine_temp,vibration_score,weather_severity,route_congestion,cargo_weight_tons,delay_minutes,delay_risk
0,EVT_00000000,2026-01-03 16:15:00,TRAIN_0101,ROUTE_007,31.877310,-118.418491,163.075311,236.888859,81.145080,97.316239,176.830474,4.358306,3,0.453440,3475.579755,33.050447,1
1,EVT_00000001,2026-01-24 05:14:00,TRAIN_0002,ROUTE_010,37.317575,-86.730921,159.386319,319.794107,85.000000,102.323842,157.690730,2.082853,2,0.334176,7180.908234,18.527557,0
2,EVT_00000002,2026-01-20 15:17:00,TRAIN_0160,ROUTE_015,36.516983,-103.260188,214.603464,310.688189,85.000000,90.288007,195.059629,0.813885,0,0.344303,8573.058800,14.455687,0
3,EVT_00000003,2026-01-14 03:59:00,TRAIN_0179,ROUTE_009,42.273409,-77.170766,216.859688,220.619067,50.866560,90.707937,168.244862,0.828496,2,0.199514,9534.519123,8.753673,0
4,EVT_00000004,2026-01-13 23:46:00,TRAIN_0061,ROUTE_001,38.560461,-90.580270,155.921078,33.060690,5.740027,85.668059,136.862996,1.567039,1,0.253572,9615.291861,14.296432,0


(5000, 17)


## 2. Run Bronze / Silver / Gold pipeline

In [3]:
from src.pipeline import bronze_ingest, silver_clean, gold_features

bronze_df = bronze_ingest(
    ROOT / "data/raw/train_telemetry_events.csv",
    ROOT / "data/bronze/telemetry_bronze.parquet"
)

silver_df = silver_clean(
    ROOT / "data/bronze/telemetry_bronze.parquet",
    ROOT / "data/silver/telemetry_silver.parquet"
)

gold_df = gold_features(
    ROOT / "data/silver/telemetry_silver.parquet",
    ROOT / "data/gold/train_delay_features.parquet"
)

display(gold_df.head())
print(gold_df.shape)

,event_id,event_time,train_id,route_id,latitude,longitude,scheduled_minutes,distance_miles,avg_speed_mph,brake_pressure,...,weather_severity,route_congestion,cargo_weight_tons,delay_minutes,delay_risk,hour,day_of_week,route_avg_delay,route_avg_congestion,route_event_count
0,EVT_00000000,2026-01-03 16:15:00,TRAIN_0101,ROUTE_007,31.877310,-118.418491,163.075311,236.888859,81.145080,97.316239,...,3,0.453440,3475.579755,33.050447,1,16,5,16.439822,0.266886,172
1,EVT_00000001,2026-01-24 05:14:00,TRAIN_0002,ROUTE_010,37.317575,-86.730921,159.386319,319.794107,85.000000,102.323842,...,2,0.334176,7180.908234,18.527557,0,5,5,15.733924,0.301572,192
2,EVT_00000002,2026-01-20 15:17:00,TRAIN_0160,ROUTE_015,36.516983,-103.260188,214.603464,310.688189,85.000000,90.288007,...,0,0.344303,8573.058800,14.455687,0,15,1,15.858145,0.280049,158
3,EVT_00000003,2026-01-14 03:59:00,TRAIN_0179,ROUTE_009,42.273409,-77.170766,216.859688,220.619067,50.866560,90.707937,...,2,0.199514,9534.519123,8.753673,0,3,2,15.913714,0.302372,190
4,EVT_00000004,2026-01-13 23:46:00,TRAIN_0061,ROUTE_001,38.560461,-90.580270,155.921078,33.060690,5.740027,85.668059,...,1,0.253572,9615.291861,14.296432,0,23,1,15.583112,0.291508,163


(5000, 22)


## 3. Train neural network predictive timing model

In [4]:
from src.model import train_model

metrics = train_model(
    gold_path=ROOT / "data/gold/train_delay_features.parquet",
    model_path=ROOT / "outputs/models/delay_timing_nn.pt",
    scaler_path=ROOT / "outputs/models/scaler.joblib",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    report_path=ROOT / "outputs/tables/classification_report.csv",
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    epochs=30,
)

metrics

Epoch 01/30 | Loss=7.0272
Epoch 02/30 | Loss=7.0036
Epoch 03/30 | Loss=6.9828
Epoch 04/30 | Loss=6.9603
Epoch 05/30 | Loss=6.9389
Epoch 06/30 | Loss=6.9170
Epoch 07/30 | Loss=6.8946
Epoch 08/30 | Loss=6.8736
Epoch 09/30 | Loss=6.8526
Epoch 10/30 | Loss=6.8301
Epoch 11/30 | Loss=6.8071
Epoch 12/30 | Loss=6.7859
Epoch 13/30 | Loss=6.7635
Epoch 14/30 | Loss=6.7407
Epoch 15/30 | Loss=6.7176
Epoch 16/30 | Loss=6.6949
Epoch 17/30 | Loss=6.6718
Epoch 18/30 | Loss=6.6481
Epoch 19/30 | Loss=6.6222
Epoch 20/30 | Loss=6.5970
Epoch 21/30 | Loss=6.5714
Epoch 22/30 | Loss=6.5447
Epoch 23/30 | Loss=6.5179
Epoch 24/30 | Loss=6.4901
Epoch 25/30 | Loss=6.4629
Epoch 26/30 | Loss=6.4319
Epoch 27/30 | Loss=6.4009
Epoch 28/30 | Loss=6.3722
Epoch 29/30 | Loss=6.3405
Epoch 30/30 | Loss=6.3103


{'accuracy': 0.7072,
 'f1': 0.0,
 'auc': 0.5547158964468512,
 'mae_delay_minutes': 14.985297203063965,
 'rmse_delay_minutes': 16.70799979291591,
 'n_test': 1250}

## 4. Generate visual outputs

In [5]:
from src.visualization import generate_figures

figures = generate_figures(
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    loss_path=ROOT / "outputs/tables/training_loss.csv",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    output_dir=ROOT / "outputs/figures",
)

figures

[PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/precision_recall_delay_risk.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/roc_curve_delay_risk.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/confusion_matrix_delay_risk.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/training_loss_curve.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/actual_vs_predicted_delay_minutes.png'),
 PosixPath('/Users/yuzhang/projects/Machine_learning/01_transportation_telemetry_predictive_timing/outputs/figures/classification_metrics_bar_chart.png')]

## 5. Review prediction outputs

In [6]:
import pandas as pd

pred = pd.read_csv(ROOT / "outputs/tables/predictions.csv")
report = pd.read_csv(ROOT / "outputs/tables/classification_report.csv")

display(pred.head())
display(report)

,actual_delay_risk,predicted_delay_risk,predicted_delay_probability,actual_delay_minutes,predicted_delay_minutes
0,1.0,0,0.390185,21.801655,0.865485
1,1.0,0,0.382947,29.257046,1.257421
2,1.0,0,0.403940,24.140720,0.613990
3,0.0,0,0.380540,18.370869,0.855477
4,0.0,0,0.387539,16.171385,1.059852


,Unnamed: 0,precision,recall,f1-score,support
0,0.0,0.707200,1.0000,0.828491,884.0000
1,1.0,0.000000,0.0000,0.000000,366.0000
2,accuracy,0.707200,0.7072,0.707200,0.7072
3,macro avg,0.353600,0.5000,0.414246,1250.0000
4,weighted avg,0.500132,0.7072,0.585909,1250.0000


## Final Interpretation

This project shows a local, runnable version of a transportation telemetry platform. It demonstrates how real-time sensor events can move through a medallion lakehouse pipeline and feed a neural network model that predicts delay risk and delay minutes.

This is a synthetic portfolio project. In production, it would connect to Kafka/Kinesis, Spark/Flink, Delta/Iceberg, cloud storage, orchestration, monitoring, and secure enterprise governance.
